In [1]:
# Ipython...
from __future__ import print_function, division
import matplotlib.pyplot as plt
from sympy.physics.vector import init_vprinting
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
get_ipython().run_line_magic('matplotlib', 'inline')
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')
init_vprinting(use_latex='mathjax', pretty_print=False)
# Sympy
import numpy as np
import sympy as sp
from sympy import Matrix, symbols, simplify, Function, expand_trig, Symbol, diff
from sympy import cos, sin, transpose, pi
from sympy import latex, python
from sympy.physics.mechanics import dynamicsymbols, ReferenceFrame, Point, inertia
# Yams sympy
from welib.yams.yams_sympy       import YAMSRigidBody, YAMSInertialBody, YAMSFlexibleBody
from welib.yams.yams_sympy_model import YAMSModel


In [2]:
# Ipython...
from __future__ import print_function, division
import matplotlib.pyplot as plt
from sympy.physics.vector import init_vprinting
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
get_ipython().run_line_magic('matplotlib', 'inline')
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')
init_vprinting(use_latex='mathjax', pretty_print=False)
# Sympy
import numpy as np
import sympy as sp
from sympy import Matrix, symbols, simplify, Function, expand_trig, Symbol, diff
from sympy import cos, sin, transpose, pi
from sympy import latex, python
from sympy.physics.mechanics import dynamicsymbols, ReferenceFrame, Point, inertia
# Yams sympy
from welib.yams.yams_sympy       import YAMSRigidBody, YAMSInertialBody, YAMSFlexibleBody
from welib.yams.yams_sympy_model import YAMSModel


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# -- Degrees of freedom
time = dynamicsymbols._t
theta, psi = dynamicsymbols('theta, psi')
thetad, psid = dynamicsymbols('thetad, psid')

# --- Reference frame
ref = YAMSInertialBody('E') 

# Rigid Nacelle
nac = YAMSRigidBody('N', rho_G=[Symbol('x_NG'),0,Symbol('z_NG')], J_form='diag') 

# Flexible Blade (using the same syntax as your tower, but treating it as a blade)
bld = YAMSFlexibleBody('B', nq=1, directions=['x'], orderMM=1, orderH=1, predefined_kind='bld-z') 

bodies = [nac, bld]

# --- Connections
ref.connectTo(nac, type='Joint', rel_pos=(0, 0, 0), rot_type='Body', rot_amounts=(0,theta,0), rot_order='XYZ')
nac.connectTo(bld, type='Joint', rel_pos=(Symbol('x_NB'), 0, Symbol('z_NB')), rot_type='Body', rot_amounts=(psi,0,0), rot_order='XYZ')

In [12]:
# --- Flexible Body Substitutions
mySubs = []
mySubs += [(bld.ucList[0], 0)]
mySubs += [(bld.vcList[0], 1)]

In [13]:
# --- DOFs and kinetic equations
coordinates = [theta, psi, bld.q[0]]
speeds      = [thetad, psid, bld.qd[0]]

kdeqsSubs = []
kdeqsSubs += [(thetad, sp.diff(theta, time))]
kdeqsSubs += [(psid, sp.diff(psi, time))]
kdeqsSubs += [(bld.qd[0], bld.qdot[0])]

print('coordinates:', coordinates)
print('kdeqsSubs  :', kdeqsSubs)

coordinates: [theta(t), psi(t), q_B1(t)]
kdeqsSubs  : [(thetad(t), Derivative(theta(t), t)), (psid(t), Derivative(psi(t), t)), (qd_B1(t), Derivative(q_B1(t), t))]


In [14]:
# --- Loads
gravity = Symbol('g')
body_loads = []
for bdy in bodies:
    body_loads += [(bdy, (bdy.masscenter, -bdy.mass * gravity * ref.frame.z))]  

model = YAMSModel(name='NTB', ref=ref, bodies=bodies, body_loads=body_loads, 
                  coordinates=coordinates, speeds=speeds, kdeqsSubs=kdeqsSubs, g_vect=-gravity*ref.frame.z)

model.kaneEquations(Mform='symbolic') # or 'TaylorExpanded' depending on your preference
extraSubs = mySubs

EOM = model.to_EOM(extraSubs=extraSubs)
EOM.mass_forcing_form()

>>> yams_simpy, TODO simplifications of mass matrix for blade
>>> yams_simpy, TODO simplifications of mass matrix for blade


In [16]:
print('Mass matrix:')
EOM.M
print('Forcing term:')
EOM.F

Mass matrix:


Matrix([
[J_Byy*cos(psi)**2 - 2*J_Byz*sin(psi)*cos(psi) + J_Bzz*sin(psi)**2 + J_yy_N + M_B*x_NB**2*sin(psi)**2 + M_B*x_NB**2*cos(psi)**2 + M_B*z_NB**2 + 2*M_B15*z_NB*cos(psi) - 2*M_B16*z_NB*sin(psi) + 2*M_B26*x_NB*sin(psi)**2 - 2*M_B35*x_NB*cos(psi)**2 + M_N*x_NG**2 + M_N*z_NG**2, J_Bxy*sin(theta)**2*cos(psi) + J_Bxy*cos(psi)*cos(theta)**2 - J_Bxz*sin(psi)*sin(theta)**2 - J_Bxz*sin(psi)*cos(theta)**2 - M_B24*x_NB*sin(psi)*sin(theta)**2 - M_B24*x_NB*sin(psi)*cos(theta)**2 - M_B34*x_NB*sin(theta)**2*cos(psi) - M_B34*x_NB*cos(psi)*cos(theta)**2, M_B17*z_NB - M_B27*x_NB*sin(psi) - M_B37*x_NB*cos(psi) + M_B57*cos(psi) - M_B67*sin(psi)],
[                                                                                                                                                                                                   J_Bxy*cos(psi) - J_Bxz*sin(psi) - M_B24*x_NB*sin(psi) - M_B34*x_NB*cos(psi),                                                                                        

Forcing term:


Matrix([
[-2*C_t^0_B_1x*x_NB*sin(psi)**2*q_B1'*theta' - 2*C_t^0_B_1x*x_NB*cos(psi)**2*q_B1'*theta' + 2*C_t^0_B_1y*x_NB*cos(psi)*psi'*q_B1' - 2*C_t^0_B_1y*z_NB*sin(psi)*q_B1'*theta' - 2*C_t^0_B_1z*x_NB*sin(psi)*psi'*q_B1' - 2*C_t^0_B_1z*z_NB*cos(psi)*q_B1'*theta' - G_r_1^0_B_yx*cos(psi)*psi'*q_B1' - G_r_1^0_B_yy*cos(psi)**2*q_B1'*theta' + G_r_1^0_B_yz*sin(psi)*cos(psi)*q_B1'*theta' + G_r_1^0_B_zx*sin(psi)*psi'*q_B1' + G_r_1^0_B_zy*sin(psi)*cos(psi)*q_B1'*theta' - G_r_1^0_B_zz*sin(psi)**2*q_B1'*theta' + J^0_B_yx*sin(psi)*psi'**2 + J^0_B_yy*sin(psi)*cos(psi)*psi'*theta' - J^0_B_yz*sin(psi)**2*psi'*theta' + J^0_B_zx*cos(psi)*psi'**2 + J^0_B_zy*cos(psi)**2*psi'*theta' - J^0_B_zz*sin(psi)*cos(psi)*psi'*theta' + J_Byy*sin(psi)*sin(theta)**2*cos(psi)*psi'*theta' + J_Byy*sin(psi)*cos(psi)*cos(theta)**2*psi'*theta' - J_Byz*sin(psi)**2*sin(theta)**2*psi'*theta' - J_Byz*sin(psi)**2*cos(theta)**2*psi'*theta' + J_Byz*sin(theta)**2*cos(psi)**2*psi'*theta' + J_Byz*cos(psi)**2*cos(theta)**2*psi'*theta'